In [2]:
# Same parameter pattern as NB_01
# file_name is overridden by the pipeline ForEach at runtime

file_name     = "listings_europe.json"
source_region = "Europe"
file_path     = f"Files/incoming/{file_name}"

print(f"Processing: {file_name}")
print(f"Source region: {source_region}")
print(f"Full path: {file_path}")

StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 3, Finished, Available, Finished, False)

Processing: listings_europe.json
Source region: Europe
Full path: Files/incoming/listings_europe.json


In [3]:
from pyspark.sql.functions import (
    col, to_date, trim, upper, lower,
    when, lit, current_timestamp, round as spark_round,initcap
)

# WHY spark.read.json WORKS DIFFERENTLY FROM CSV:
# JSON is self-describing — Spark infers column names and types automatically
# No schema definition needed
# The tradeoff: column names come in exactly as they are in the file
# so we get snake_case names that need renaming

df_raw =df_raw = spark.read.option("multiline", "true").json(file_path)

raw_count = df_raw.count()
print(f"Records read from {file_name}: {raw_count}")
print("\nSchema as read from JSON:")
df_raw.printSchema()

# Confirm we got the expected columns
print("\nSample row:")
df_raw.select(
    "listing_id", "city", "country_code",
    "asking_price", "listed_on", "status"
).show(3, truncate=False)

StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 4, Finished, Available, Finished, False)

Records read from listings_europe.json: 600

Schema as read from JSON:
root
 |-- agent_email: string (nullable = true)
 |-- agent_name: string (nullable = true)
 |-- area_sqm: double (nullable = true)
 |-- asking_price: long (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- last_modified: string (nullable = true)
 |-- listed_on: string (nullable = true)
 |-- listing_id: string (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- num_bathrooms: long (nullable = true)
 |-- num_bedrooms: long (nullable = true)
 |-- property_type: string (nullable = true)
 |-- status: string (nullable = true)


Sample row:
+----------+------+------------+------------+----------+------+
|listing_id|city  |country_code|asking_price|listed_on |status|
+----------+------+------------+------------+----------+------+
|EU-00001  |Madrid|ES          |6737000     |13-01-2024|active|
|EU-00002  |Madrid|ES    

In [4]:
# WHY EXPLICIT RENAME RATHER THAN SELECT:
# Renaming is more readable and safer than SELECT *
# It makes the mapping between source and unified schema explicit
# Anyone reading this notebook can see exactly what changed
#
# COLUMN MAPPING — Europe JSON → Unified schema:
# listing_id     → ListingID
# agent_name     → AgentName
# agent_email    → dropped (not in unified schema)
# property_type  → PropertyType
# city           → City
# country_code   → Country (rename — will resolve ISO2 to full name below)
# num_bedrooms   → Bedrooms
# num_bathrooms  → Bathrooms
# area_sqm       → AreaSqFt (rename — will convert sqm to sqft below)
# asking_price   → ListingPrice
# listed_on      → ListDate (rename — will parse DD-MM-YYYY below)
# status         → Status
# description    → Description
# listing_url    → ListingURL
# last_modified  → dropped

df_renamed = df_raw \
    .withColumnRenamed("listing_id",    "ListingID") \
    .withColumnRenamed("agent_name",    "AgentName") \
    .withColumnRenamed("property_type", "PropertyType") \
    .withColumnRenamed("city",          "City") \
    .withColumnRenamed("country_code",  "Country") \
    .withColumnRenamed("num_bedrooms",  "Bedrooms") \
    .withColumnRenamed("num_bathrooms", "Bathrooms") \
    .withColumnRenamed("area_sqm",      "AreaSqFt") \
    .withColumnRenamed("asking_price",  "ListingPrice") \
    .withColumnRenamed("listed_on",     "ListDate") \
    .withColumnRenamed("status",        "Status") \
    .withColumnRenamed("description",   "Description") \
    .withColumnRenamed("listing_url",   "ListingURL") \
    .drop("agent_email", "last_modified")

print("Columns after rename:")
print(df_renamed.columns)

StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 5, Finished, Available, Finished, False)

Columns after rename:
['AgentName', 'AreaSqFt', 'ListingPrice', 'City', 'Country', 'Description', 'ListDate', 'ListingID', 'ListingURL', 'Bathrooms', 'Bedrooms', 'PropertyType', 'Status']


In [5]:
# WHY THIS DATE FORMAT IS TRICKY:
# Python and Spark both default to YYYY-MM-DD or MM/DD/YYYY
# DD-MM-YYYY is the European standard but Spark will misparse it
# if you use the wrong format string
#
# IMPORTANT: Spark format strings use different tokens from Python:
# Spark: dd-MM-yyyy  (lowercase dd = day, uppercase MM = month)
# Python: %d-%m-%Y
# Using the wrong one produces silent wrong dates — April 5 becomes May 4
# Always test date parsing by checking a known value

df_dated = df_renamed \
    .withColumn("ListDate",
        to_date(col("ListDate"), "dd-MM-yyyy"))

# Verify — check a sample date looks correct
null_dates = df_dated.filter(col("ListDate").isNull()).count()
print(f"Null dates after parsing: {null_dates}  (should be 0)")

df_dated.select("ListingID", "ListDate") \
    .show(5, truncate=False)

StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 6, Finished, Available, Finished, False)

Null dates after parsing: 0  (should be 0)
+---------+----------+
|ListingID|ListDate  |
+---------+----------+
|EU-00001 |2024-01-13|
|EU-00002 |2023-11-14|
|EU-00003 |2023-05-17|
|EU-00004 |2022-01-28|
|EU-00005 |2022-03-20|
+---------+----------+
only showing top 5 rows



In [6]:
# WHY UNIT CONVERSION MATTERS:
# The USA file uses sqft. The Europe file uses sqm.
# If we store both as-is, a 100 sqm apartment appears smaller than
# a 100 sqft cupboard when sorted by area — completely wrong analysis.
# All area values must be in the same unit before reaching Silver.
#
# CONVERSION FACTOR: 1 sqm = 10.764 sqft
# This is the internationally accepted conversion factor.
# We round to 1 decimal place — sub-sqft precision is meaningless for
# property listings.

SQM_TO_SQFT = 10.764

df_converted = df_dated \
    .withColumn("AreaSqFt",
        spark_round(col("AreaSqFt") * SQM_TO_SQFT, 1))

print(f"Unit conversion: sqm → sqft (× {SQM_TO_SQFT})")
print("Sample area values after conversion:")
df_converted.select("ListingID", "AreaSqFt") \
    .show(5, truncate=False)

StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 7, Finished, Available, Finished, False)

Unit conversion: sqm → sqft (× 10.764)
Sample area values after conversion:
+---------+--------+
|ListingID|AreaSqFt|
+---------+--------+
|EU-00001 |2306.7  |
|EU-00002 |3678.1  |
|EU-00003 |1285.2  |
|EU-00004 |958.0   |
|EU-00005 |5504.7  |
+---------+--------+
only showing top 5 rows



In [7]:
# WHY ISO2 CODES NEED RESOLVING:
# Power BI maps and charts work on country names not ISO codes.
# Also the USA file uses full names — if we leave Europe as ISO codes
# the Silver table has mixed formats which breaks any country-level analysis.
#
# WHY A WHEN CHAIN AND NOT A JOIN:
# A JOIN to a lookup table adds complexity and a dependency.
# With only 6 European countries in our dataset a WHEN chain is
# readable, maintainable, and has zero performance overhead.
# For 50+ countries a lookup table would be appropriate.

df_countries = df_converted \
    .withColumn("Country",
        when(col("Country") == "GB", lit("United Kingdom"))
       .when(col("Country") == "FR", lit("France"))
       .when(col("Country") == "DE", lit("Germany"))
       .when(col("Country") == "ES", lit("Spain"))
       .when(col("Country") == "NL", lit("Netherlands"))
       .when(col("Country") == "IT", lit("Italy"))
       .when(col("Country") == "BE", lit("Belgium"))
       .when(col("Country") == "PT", lit("Portugal"))
       .when(col("Country") == "CH", lit("Switzerland"))
       .when(col("Country") == "AT", lit("Austria"))
       .otherwise(col("Country")))  # keep original if not in list

# Verify no ISO codes remain
print("Country distribution after resolution:")
df_countries.groupBy("Country").count() \
    .orderBy("count", ascending=False).show(truncate=False)

StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 8, Finished, Available, Finished, False)

Country distribution after resolution:
+--------------+-----+
|Country       |count|
+--------------+-----+
|France        |147  |
|United Kingdom|142  |
|Germany       |125  |
|Spain         |86   |
|Netherlands   |52   |
|Italy         |48   |
+--------------+-----+



In [9]:
# WHY STANDARDISE IN BRONZE FOR EUROPE:
# Europe uses lowercase: active / sold
# USA uses title case: Active / Sold
# These would appear as separate values in any GROUP BY analysis
# unless standardised before reaching Silver
#
# EUROPE STATUS MAPPING:
# active → Active
# sold   → Sold
# any other value → Other

df_standardised = df_countries \
    .withColumn("Status",
        when(lower(trim(col("Status"))) == "active", lit("Active"))
       .when(lower(trim(col("Status"))) == "sold",   lit("Sold"))
       .otherwise(lit("Other"))) \
    .withColumn("PropertyType",
        initcap(trim(col("PropertyType"))))

print("Status distribution after standardisation:")
df_standardised.groupBy("Status").count() \
    .orderBy("count", ascending=False).show()

print("Property type distribution:")
df_standardised.groupBy("PropertyType").count() \
    .orderBy("count", ascending=False).show()



StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 10, Finished, Available, Finished, False)

Status distribution after standardisation:
+------+-----+
|Status|count|
+------+-----+
|Active|  385|
|  Sold|  215|
+------+-----+

Property type distribution:
+------------+-----+
|PropertyType|count|
+------------+-----+
|    Bungalow|  110|
|       House|  109|
|       Villa|  108|
|   Townhouse|  100|
|   Penthouse|   93|
|        Flat|   80|
+------------+-----+



In [10]:
# Bedrooms and Bathrooms came in as LONG from JSON inference
# Cast to integer for consistency with the USA file

df_typed = df_standardised \
    .withColumn("Bedrooms",  col("Bedrooms").cast("integer")) \
    .withColumn("Bathrooms", col("Bathrooms").cast("integer")) \
    .withColumn("ListingPrice", col("ListingPrice").cast("double"))

print("Types after casting:")
df_typed.printSchema()

StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 11, Finished, Available, Finished, False)

Types after casting:
root
 |-- AgentName: string (nullable = true)
 |-- AreaSqFt: double (nullable = true)
 |-- ListingPrice: double (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- ListDate: date (nullable = true)
 |-- ListingID: string (nullable = true)
 |-- ListingURL: string (nullable = true)
 |-- Bathrooms: integer (nullable = true)
 |-- Bedrooms: integer (nullable = true)
 |-- PropertyType: string (nullable = true)
 |-- Status: string (nullable = false)



In [11]:
# Add source audit columns — same pattern as NB_01
df_bronze = df_typed \
    .withColumn("SourceRegion",  lit("Europe")) \
    .withColumn("SourceFile",    lit(file_name)) \
    .withColumn("_ingestion_ts", current_timestamp()) \
    .withColumn("State",         lit(None).cast("string"))
    # WHY State = null for Europe:
    # USA has State (NY, CA etc). Europe does not use states.
    # We add a null State column so the schema matches bronze_listings
    # which already has a State column from the USA rows.

# Remove existing Europe rows — idempotent rerun safety
from delta.tables import DeltaTable

if spark.catalog.tableExists("bronze_listings"):
    target = DeltaTable.forName(spark, "bronze_listings")
    target.delete("SourceRegion = 'Europe'")
    print("Removed existing Europe rows")

df_bronze.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze_listings")

europe_count = spark.sql(
    "SELECT COUNT(*) FROM bronze_listings WHERE SourceRegion = 'Europe'"
).collect()[0][0]
total = spark.table("bronze_listings").count()
print(f"Appended {europe_count} Europe rows  |  Total bronze_listings: {total}")

StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 12, Finished, Available, Finished, False)

Removed existing Europe rows
Appended 600 Europe rows  |  Total bronze_listings: 1400


In [12]:
print("=== NB_02 — Europe JSON Verification ===")

# Row count by region — should now show USA + Europe
spark.sql("""
    SELECT SourceRegion, COUNT(*) AS Rows
    FROM   bronze_listings
    GROUP BY SourceRegion
    ORDER BY Rows DESC
""").show()

# Confirm area values are in sqft range (not sqm)
# A typical EU flat: 60–80 sqm = 646–861 sqft
# If AreaSqFt shows values like 65 it means sqm was NOT converted
spark.sql("""
    SELECT
        ROUND(MIN(AreaSqFt), 0) AS MinAreaSqFt,
        ROUND(MAX(AreaSqFt), 0) AS MaxAreaSqFt,
        ROUND(AVG(AreaSqFt), 0) AS AvgAreaSqFt
    FROM bronze_listings
    WHERE SourceRegion = 'Europe'
""").show()

# Confirm no ISO2 codes remain
spark.sql("""
    SELECT Country, COUNT(*) AS Count
    FROM   bronze_listings
    WHERE  SourceRegion = 'Europe'
    GROUP BY Country
    ORDER BY Count DESC
""").show(truncate=False)

# Confirm dates parsed correctly
spark.sql("""
    SELECT
        MIN(ListDate) AS EarliestListing,
        MAX(ListDate) AS LatestListing
    FROM bronze_listings
    WHERE SourceRegion = 'Europe'
""").show()

# Sample rows
spark.table("bronze_listings") \
    .filter("SourceRegion = 'Europe'") \
    .select("ListingID", "City", "Country",
            "Bedrooms", "AreaSqFt", "ListingPrice",
            "ListDate", "Status") \
    .show(5, truncate=False)

StatementMeta(, 77ba5a9e-39ff-453b-bab6-4d469a33da43, 13, Finished, Available, Finished, False)

=== NB_02 — Europe JSON Verification ===
+------------+----+
|SourceRegion|Rows|
+------------+----+
|         USA| 800|
|      Europe| 600|
+------------+----+

+-----------+-----------+-----------+
|MinAreaSqFt|MaxAreaSqFt|AvgAreaSqFt|
+-----------+-----------+-----------+
|      454.0|     6994.0|     3568.0|
+-----------+-----------+-----------+

+--------------+-----+
|Country       |Count|
+--------------+-----+
|France        |147  |
|United Kingdom|142  |
|Germany       |125  |
|Spain         |86   |
|Netherlands   |52   |
|Italy         |48   |
+--------------+-----+

+---------------+-------------+
|EarliestListing|LatestListing|
+---------------+-------------+
|     2022-01-02|   2024-05-31|
+---------------+-------------+

+---------+----------+--------------+--------+--------+------------+----------+------+
|ListingID|City      |Country       |Bedrooms|AreaSqFt|ListingPrice|ListDate  |Status|
+---------+----------+--------------+--------+--------+------------+----------+--